# KTO — Kahneman-Tversky Optimization
### Direct / Reward-Free Alignment  ·  Colab T4 (16 GB) ready

> **DPO/IPO need matched pairs** — every prompt must carry *both* a chosen and a rejected completion. That data is expensive and rare.
> **KTO needs only a thumbs up/down on a single output.** It aligns on the cheap, fragmented binary feedback you already collect in production.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)
- **KTO (Kahneman-Tversky Optimization)** (*Ethayarajh et al., "KTO: Model Alignment as Prospect Theoretic Optimization"*) is a **HALO** — a **Human-Aware LOss** — that aligns a model using **unpaired binary labels**: each example is `(prompt, completion, label ∈ {desirable, undesirable})`, i.e. a single **👍 / 👎**.
- It is grounded in **prospect theory** (Kahneman & Tversky): humans judge outcomes as **gains/losses relative to a reference point**, with a value function that is **loss-averse** (losses hurt more than equal gains help).
- KTO reuses the same **implicit reward** as DPO/IPO — the β-scaled policy-vs-reference log-ratio — but instead of contrasting two completions, it pushes each completion's reward through a **prospect-theory value function** centered on a **KL-based reference point** `z_ref`:
  $$r_\theta(x,y) = \beta\,\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}, \qquad z_{\text{ref}} \approx \mathbb{E}_x\big[\mathrm{KL}\big(\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big)\big]$$

### One-sentence definition of the mechanics
> **KTO maximizes the prospect-theoretic *utility* of each generation's implicit reward measured against a batch-estimated KL reference point — raising the reward of 👍 outputs and lowering it for 👎 outputs, with a loss-aversion asymmetry — using only per-example binary labels instead of chosen/rejected pairs.**

The loss (per example, with desirable/undesirable weights `λ_D`, `λ_U`):
$$\mathcal{L}_{\text{KTO}} = \begin{cases} \lambda_D\big(1 - \sigma(\beta\,(r_\theta - z_{\text{ref}}))\big) & \text{if } y \text{ is desirable (👍)} \\[4pt] \lambda_U\big(1 - \sigma(\beta\,(z_{\text{ref}} - r_\theta))\big) & \text{if } y \text{ is undesirable (👎)} \end{cases}$$

### The exact engineering problem it solves
- **The pairing bottleneck.** DPO/IPO require, *for the same prompt*, a **matched** `(chosen, rejected)` pair. Collecting that means showing annotators two completions and forcing a comparison — **slow, costly, and often impossible** to reconstruct from real usage.
- **Production feedback is unpaired and fragmented.** Real signals are single thumbs-up/down clicks, 👍/👎 reactions, deletions, regenerations, support-ticket resolutions — **one label per output**, never neatly paired.
- **KTO consumes exactly that.** It drops the pairing requirement entirely, so you can align on **cheap, abundant, imbalanced** binary data — even data where a prompt has *only* a 👍 or *only* a 👎, and where the counts of positives and negatives are wildly uneven.

---

### The Human Element — Hugging Face datasets for KTO

| HF path | What it is | Why it's shaped this way for KTO |
|---|---|---|
| **`trl-lib/kto-mix-14k`** | The **official TRL KTO** dataset (~14k rows) in native KTO schema: **`prompt`, `completion`, `label` (bool)**. | This *is* the unpaired format — one completion, one boolean 👍/👎. No pairs. Used directly in Section 3. Roughly class-balanced, so default weights work. |
| **`Anthropic/hh-rlhf`** | Human `(chosen, rejected)` pairs. | **Unbundled** into KTO form: each `chosen` → `(prompt, completion, label=True)`, each `rejected` → `(prompt, completion, label=False)`. Demonstrates KTO can ingest *any* preference data as **two independent binary signals** — the pair is optional, not required. |
| **`HuggingFaceH4/ultrafeedback_binarized`** | GPT-4-scored `chosen`/`rejected` (the DPO/IPO corpus). | Same unbundling: split every pair into two labeled examples. Lets you A/B **KTO vs DPO/IPO on identical underlying data** to isolate the objective's effect. |

**Why the schema differs from DPO/IPO:** DPO/IPO's loss is a *contrast between two completions of one prompt*, so it **must** have pairs. KTO's loss compares **one completion against a global reference point** (`z_ref`), so it needs only `(prompt, completion, label)` — the pairing is gone.

> This notebook trains on **`trl-lib/kto-mix-14k`** with TRL's **`KTOTrainer`** (Section 3).

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the prospect-theory reason KTO works
- **Prospect theory's value function** is concave over gains, convex over losses, and **steeper for losses than gains** (loss aversion). KTO implements this as a **logistic value function** applied to the implicit reward `r_θ`, **shifted by the reference point `z_ref`** and **weighted asymmetrically** by `λ_D` (desirable) and `λ_U` (undesirable).
- **The reference point replaces the pair.** DPO/IPO form a margin `r_θ(y_w) − r_θ(y_l)` — they *need* two completions. KTO instead asks: *"is this completion's reward above or below the model's own average behavior, `z_ref = E[KL(π_θ‖π_ref)]`?"* That single scalar reference point is the "other side" of the comparison, so **no paired example is required**.
- **`z_ref` is estimated per batch.** TRL pairs each prompt `x` with a *mismatched* completion `y'` from elsewhere in the batch to Monte-Carlo-estimate the KL. This is why **batch size matters** for KTO: a bigger batch = a less noisy reference point.
- **Loss aversion as a knob.** Setting `λ_U > λ_D` makes 👎 examples penalize harder than 👍 examples reward — directly encoding "avoid bad output" over "seek good output," useful for safety-critical alignment.

#### VRAM & Compute Impact
- **Lighter per example than DPO/IPO on the target pass** — KTO tokenizes **one completion**, not a `chosen ⧺ rejected` pair, so the primary forward is ~half the sequence footprint. (It adds a **no-grad KL forward** on the mismatched completion, so total compute is comparable, but the *trainable* activations are smaller.)
- **Still 1 policy + 1 reference**; with **PEFT/LoRA** the reference is the base weights with **adapters disabled** → **effectively one model in VRAM**.
- **No reward model, no value model, no rollouts** — same reward-free savings as DPO/IPO vs PPO's ~4 models.
- **The real win is data economics, not FLOPs:** 1 label per sample, harvestable from live traffic, vs the annotation cost of constructing matched pairs.

#### Pros & Cons

**Pros**
- **Kills the pairing bottleneck** — aligns on cheap, abundant 👍/👎 signals; no matched completions needed.
- **Handles imbalance & fragmentation** — works when positives and negatives are uneven, or when a prompt has only one label.
- **Competitive-to-better than DPO** at scale in the KTO paper, and **more robust to noisy/extreme** labels than vanilla DPO.
- **Same reward-free, low-VRAM footprint** — drop-in TRL trainer, QLoRA-friendly.

**Cons**
- **Class-balance sensitivity** — you must tune `desirable_weight` / `undesirable_weight`; TRL recommends the **weighted** ratio `(λ_D·n_D)/(λ_U·n_U)` sit in **[1, 4/3]** (or its reciprocal) and warns otherwise.
- **Batch-size-dependent** — the `z_ref` KL estimate degrades with tiny batches.
- **More hyperparameters** — `β`, `λ_D`, `λ_U` all interact.
- **Weaker per-signal information** — a lone 👍/👎 says less than a direct A-vs-B comparison; you typically need **more** examples to match paired-method quality. Still **off-policy** (no exploration).

#### Metrics to watch (TRL `KTOTrainer` logs KTO-specific keys)
- **`kl`** — the estimated reference point `z_ref`. **KTO-distinctive**; should stay bounded, not explode.
- **`rewards/chosen`** (desirable) vs **`rewards/rejected`** (undesirable) — the former should rise **above** `kl`, the latter fall **below** it.
- **`rewards/margins`** — desirable minus undesirable; should widen then stabilize.
- **`count/chosen`, `count/rejected`** — per-batch class counts; confirm your weighting keeps them balanced enough.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit) + TRL `KTOTrainer` + the adapter-disabling reference trick.**

> ⚙️ **Different trainer, different data:** KTO uses **`KTOTrainer` / `KTOConfig`** and **unpaired** `{prompt, completion, label}` data — not `DPOTrainer` and not `(chosen, rejected)` pairs.

> ⚙️ **Reference for free:** `ref_model=None` on a **PEFT** policy makes TRL compute `z_ref` and the reward by **disabling the LoRA adapters** — the frozen 4-bit base weights *are* `π_ref`.

**Executable pipeline:**

| Step | What | Notes |
|---|---|---|
| 1 | 4-bit `Qwen2.5-0.5B-Instruct` + LoRA + tokenizer | policy **and** (adapters-off) reference |
| 2 | `trl-lib/kto-mix-14k` → `(prompt, completion, label:bool)` | **unpaired** binary signals |
| 3 | `KTOConfig` (β, desirable/undesirable weights, memory flags) | prospect-theory objective |
| 4 | `KTOTrainer.train()` | value-function loss around `z_ref` |
| 5 | Save adapter · export · inference | ship it |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch, os, math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import KTOTrainer, KTOConfig  # KTO has its own trainer/config (not DPOTrainer)

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# The Colab T4 is Turing (compute capability 7.5). bfloat16 TENSOR CORES only exist on
# Ampere (SM 8.0) and newer. Forcing bf16 here makes the bitsandbytes 4-bit dequant path
# return garbage, which becomes NaN logits -> NaN softmax -> "CUDA error: device-side
# assert triggered" inside torch.multinomial at generation time. Detect, don't assume.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8                                   # True on A100/L4/H100, False on T4
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

### Step 1 — Quantization, LoRA & Tokenizer

Start from the **Instruct** checkpoint (SFT done): it is both the **policy** and the reference `π_ref`. One 4-bit base + one LoRA config serves both (adapters on = policy, adapters off = reference). *(Setup is identical to the DPO/IPO notebooks — KTO differs in the trainer, loss, and data.)*

In [ ]:
# The SFT/Instruct start point = the KTO reference policy (pi_ref).
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit NF4: base weights (shared by policy AND reference) live in 4-bit on the T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # KTO pads batched completions; Qwen needs a pad id

# LoRA = the ONLY trainable tensors. Disabling these adapters reproduces pi_ref exactly.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

policy_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # mem-efficient attention (T4 has no FlashAttn-2)
)
policy_model.config.use_cache = False  # required with gradient checkpointing
print(policy_model.get_memory_footprint() / 1e9, "GB base (4-bit)")

### Step 2 — Dataset: `trl-lib/kto-mix-14k` (unpaired binary)

The native KTO schema: **`prompt`**, **`completion`**, **`label`** (a bool 👍/👎) — **no pairs**. Both `prompt` and `completion` are conversational message lists; we flatten to the **standard** string form (templated prompt + raw completion text) so TRL doesn't re-template.

In [ ]:
raw = load_dataset("trl-lib/kto-mix-14k", split="train")
raw = raw.shuffle(seed=42).select(range(4000))  # subset so a T4 finishes in a few minutes

def to_kto_row(ex):
    # kto-mix-14k: prompt = [{user}], completion = [{assistant}], label = bool.
    return {
        # Templated prompt ending in the assistant header the completion answers.
        "prompt":     tokenizer.apply_chat_template(ex["prompt"], tokenize=False,
                                                    add_generation_prompt=True),
        "completion": ex["completion"][-1]["content"],  # the single output text (raw)
        "label":      bool(ex["label"]),                # True = desirable (thumbs up)
    }

# String prompt/completion + bool label => TRL treats it as STANDARD KTO format.
kto_ds = raw.map(to_kto_row, remove_columns=raw.column_names)

# Class balance drives the desirable/undesirable weighting (see next cell).
n_pos = sum(kto_ds["label"]); n_neg = len(kto_ds) - n_pos
print(kto_ds)
print(f"desirable(True)={n_pos}  undesirable(False)={n_neg}  ratio={n_pos/max(n_neg,1):.2f}")
print("\n--- prompt[0] ---\n", kto_ds[0]["prompt"][:300])
print("--- completion[0] ---\n", kto_ds[0]["completion"][:200], "\nlabel:", kto_ds[0]["label"])

In [ ]:
# Step 2.5 — Free leftover GPU memory (run before training)

import gc
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB  (expect several GB free before training)")

### Step 3 — `KTOConfig` (the objective) & `KTOTrainer`

**β** scales the reward around `z_ref`. **`desirable_weight` / `undesirable_weight`** are the prospect-theory loss-aversion knobs — tune them so the *weighted* class ratio `(λ_D·n_D)/(λ_U·n_U)` lands in **[1, 4/3]**. `kto-mix-14k` is ~balanced, so we start at `1.0 / 1.0`.

In [ ]:
"""
# Compute warmup steps dynamically

`warmup_steps` is a step count, not a ratio — derive it from the actual optimizer-step total
(`rows * epochs / (batch * grad_accum)`) so it scales automatically if you change the subset size,
epoch count, or batch settings above. **~10% warmup** is the standard preference-tuning default.
"""

per_device_train_batch_size = 1
gradient_accumulation_steps = 16
num_train_epochs = 1

# Optimizer steps = passes through the data / effective batch size (world_size=1 on a single T4).
total_optimizer_steps = math.ceil(len(kto_ds) * num_train_epochs / (per_device_train_batch_size * gradient_accumulation_steps))
warmup_steps = max(1, math.ceil(total_optimizer_steps * 0.1))  # ~10% warmup
print(f"optimizer steps: {total_optimizer_steps}  ->  warmup_steps: {warmup_steps}")

In [ ]:
kto_config = KTOConfig(
    output_dir="./kto_output",
    run_name="kto-t4",

    # ---- The KTO knobs ----
    beta=0.1,                 # scales the implicit reward around the z_ref reference point
    desirable_weight=1.0,     # lambda_D: weight on 👍 examples
    undesirable_weight=1.0,   # lambda_U: weight on 👎 examples. Raise it > lambda_D to be more
                              # loss-averse (penalize bad harder). Keep (lambda_D*n_D)/(lambda_U*n_U)
                              # within [1, 4/3]; adjust here if your data is class-imbalanced.

    # ---- Sequence budget (KTO tokenizes ONE completion per example) ----
    max_length=512,          # cap on prompt + completion
    max_prompt_length=256,    # prompt-only cap

    # ---- T4 16 GB hardening ----
    per_device_train_batch_size=per_device_train_batch_size,  # T4-safe. NOTE: KTO estimates z_ref (KL) PER micro-batch,
                                     # so a smaller batch makes it noisier — raise if you have VRAM.
    gradient_accumulation_steps=gradient_accumulation_steps,  # effective batch = 16 (grad-accum, not the z_ref batch)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16, fp16=not USE_BF16,  # match the GPU: fp16 on T4, bf16 on Ampere+
    optim="paged_adamw_8bit",

    # ---- Optimization schedule ----
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    num_train_epochs=num_train_epochs,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# ref_model=None + peft_config => TRL builds pi_ref (and the z_ref KL estimate) by DISABLING
# the LoRA adapters. ONE set of 4-bit base weights is BOTH policy and reference.
kto_trainer = KTOTrainer(
    model=policy_model,
    ref_model=None,
    args=kto_config,
    train_dataset=kto_ds,
    processing_class=tokenizer,  # TRL >= 0.12 (was `tokenizer=` on older versions)
    peft_config=peft_config,
)

### Step 4 — Train

Watch the **`kl`** metric (the `z_ref` reference point) stay bounded while **`rewards/chosen`** climbs above it and **`rewards/rejected`** drops below — that separation around the reference point *is* KTO working.

In [ ]:
kto_trainer.train()

# Save the aligned LoRA adapter (a few MB, not GB).
kto_trainer.save_model("./kto_aligned_adapter")
tokenizer.save_pretrained("./kto_aligned_adapter")

## Export — Download the Aligned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./kto_aligned_adapter"
output_filename = "kto_aligned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — Evaluate the KTO-Aligned Policy

Reload the **Instruct base + KTO adapter** and generate with the **same chat template** used in training (format must match or an Instruct model produces junk regardless of alignment quality).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Re-derive the dtype here so this cell works standalone after a restart.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./kto_aligned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_path)  # attach the KTO adapter
model.eval()

In [ ]:
# Same chat template the KTO loop used (single user turn).
def generate_response(user_prompt, max_new_tokens=256, temperature=0.7):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Decode only the newly generated completion (slice off the prompt tokens).
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "Why can KTO align a model without paired chosen/rejected data?",
    "My friend has been feeling really down lately. How can I support them?",
]

print("--- KTO-Aligned Responses ---")
for i, p in enumerate(test_prompts, 1):
    print(f"\n[Prompt {i}]: {p}")
    print(f"[Response]: {generate_response(p).strip()}")
    print("-" * 60)